In [1]:
# ============================================================
# CODSOFT DATA SCIENCE INTERNSHIP
# PROJECT: MOVIE RATING PREDICTION
# DATASET: IMDb Movies India
# ============================================================

print("CodSoft Data Science Internship")
print("Project: Movie Rating Prediction")
print("Dataset: IMDb Movies India")

CodSoft Data Science Internship
Project: Movie Rating Prediction
Dataset: IMDb Movies India


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
from google.colab import files

uploaded = files.upload()

Saving IMDb Movies India.csv to IMDb Movies India (1).csv


In [6]:
df = pd.read_csv("IMDb Movies India.csv", encoding="latin1")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [7]:
df = pd.read_csv("IMDb Movies India(1).csv", encoding="latin1")

FileNotFoundError: [Errno 2] No such file or directory: 'IMDb Movies India(1).csv'

In [8]:
df.head()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali


In [9]:
df.tail()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
15504,Zulm Ko Jala Doonga,(1988),NaN,Action,4.6,11,Mahendra Shah,Naseeruddin Shah,Sumeet Saigal,Suparna Anand
15505,Zulmi,(1999),129 min,"Action, Drama",4.5,655,Kuku Kohli,Akshay Kumar,Twinkle Khanna,Aruna Irani
15506,Zulmi Raj,(2005),NaN,Action,NaN,NaN,Kiran Thej,Sangeeta Tiwari,NaN,NaN
15507,Zulmi Shikari,(1988),NaN,Action,NaN,NaN,NaN,NaN,NaN,NaN
15508,Zulm-O-Sitam,(1998),130 min,"Action, Drama",6.2,20,K.C. Bokadia,Dharmendra,Jaya Prada,Arjun Sarja


In [10]:
df.shape

(15509, 10)

In [11]:
df.columns

Index(['Name', 'Year', 'Duration', 'Genre', 'Rating', 'Votes', 'Director',
       'Actor 1', 'Actor 2', 'Actor 3'],
      dtype='object')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  object 
 1   Year      14981 non-null  object 
 2   Duration  7240 non-null   object 
 3   Genre     13632 non-null  object 
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   object 
 6   Director  14984 non-null  object 
 7   Actor 1   13892 non-null  object 
 8   Actor 2   13125 non-null  object 
 9   Actor 3   12365 non-null  object 
dtypes: float64(1), object(9)
memory usage: 1.2+ MB


In [13]:
df.describe(include="all")

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
count,15509,14981,7240,13632,7919.000000,7920,14984,13892,13125,12365
unique,13838,102,182,485,NaN,2034,5938,4718,4891,4820
top,Mamta,(2019),120 min,Drama,NaN,8,Jayant Desai,Ashok Kumar,Rekha,Pran
freq,7,410,240,2780,NaN,227,58,158,83,91
mean,NaN,NaN,NaN,NaN,5.841621,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,1.381777,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,1.100000,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,4.900000,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,6.000000,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,6.800000,NaN,NaN,NaN,NaN,NaN


In [14]:
df.isnull().sum()

,0
Name,0
Year,528
Duration,8269
Genre,1877
Rating,7590
Votes,7589
Director,525
Actor 1,1617
Actor 2,2384
Actor 3,3144


In [15]:
missing_percentage = df.isnull().mean() * 100

missing_percentage.sort_values(ascending=False)

,0
Duration,53.317429
Rating,48.939326
Votes,48.932878
Actor 3,20.272100
Actor 2,15.371720
Genre,12.102650
Actor 1,10.426204
Year,3.404475
Director,3.385131
Name,0.000000


In [16]:
print("Duplicate rows before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate rows after:", df.duplicated().sum())

Duplicate rows before: 6
Duplicate rows after: 0


In [18]:
df["Year"] = df["Year"].astype(str).str.extract(r"(\d{4})")[0]
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

In [19]:
df["Year"].head()

,Year
0,NaN
1,2019.0
2,2021.0
3,2019.0
4,2010.0


In [21]:
df["Duration"] = df["Duration"].astype(str).str.extract(r"(\d+)")[0]
df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")

In [22]:
df["Duration"].head()

,Duration
0,NaN
1,109.0
2,90.0
3,110.0
4,105.0


In [23]:
df["Votes"] = df["Votes"].astype(str).str.replace(",", "", regex=False)

df["Votes"] = pd.to_numeric(df["Votes"], errors="coerce")

In [24]:
df["Votes"].head()

,Votes
0,NaN
1,8.0
2,NaN
3,35.0
4,NaN


In [25]:
df["Rating"].describe()

,Rating
count,7919.000000
mean,5.841621
std,1.381777
min,1.100000
25%,4.900000
50%,6.000000
75%,6.800000
max,10.000000


In [26]:
model_df = df.dropna(subset=["Rating"]).copy()

print("Rows available for modeling:", len(model_df))

Rows available for modeling: 7919


In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(model_df["Rating"], bins=20, kde=True)

plt.title("Distribution of IMDb Movie Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Movies")

plt.show()

In [ ]:
genre_counts = (
    model_df["Genre"]
    .dropna()
    .str.split(", ")
    .explode()
    .value_counts()
)

genre_counts.head(15)

In [ ]:
plt.figure(figsize=(12, 6))

genre_counts.head(15).plot(kind="bar")

plt.title("Top 15 Movie Genres")
plt.xlabel("Genre")
plt.ylabel("Number of Movies")
plt.xticks(rotation=45)

plt.show()

In [ ]:
genre_rating = (
    model_df
    .dropna(subset=["Genre"])
    .assign(Genre=model_df["Genre"].str.split(", "))
    .explode("Genre")
    .groupby("Genre")["Rating"]
    .mean()
    .sort_values(ascending=False)
)

genre_rating.head(15)

In [ ]:
plt.figure(figsize=(12, 6))

genre_rating.head(15).plot(kind="bar")

plt.title("Average IMDb Rating by Genre")
plt.xlabel("Genre")
plt.ylabel("Average Rating")
plt.xticks(rotation=45)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    model_df["Votes"],
    model_df["Rating"],
    alpha=0.3
)

plt.title("IMDb Rating vs Number of Votes")
plt.xlabel("Number of Votes")
plt.ylabel("Rating")

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    np.log1p(model_df["Votes"]),
    model_df["Rating"],
    alpha=0.3
)

plt.title("IMDb Rating vs Log(Number of Votes)")
plt.xlabel("Log(Number of Votes)")
plt.ylabel("Rating")

plt.show()

In [ ]:
year_rating = (
    model_df
    .groupby("Year")["Rating"]
    .mean()
    .dropna()
)

plt.figure(figsize=(12, 6))

plt.plot(year_rating.index, year_rating.values)

plt.title("Average Movie Rating by Year")
plt.xlabel("Year")
plt.ylabel("Average Rating")

plt.show()

In [ ]:
model_df[
    ["Name", "Year", "Genre", "Rating", "Votes", "Director"]
].sort_values(
    by="Rating",
    ascending=False
).head(20)

In [ ]:
features = [
    "Year",
    "Duration",
    "Votes",
    "Genre",
    "Director",
    "Actor 1",
    "Actor 2",
    "Actor 3"
]

X = model_df[features]
y = model_df["Rating"]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
numeric_features = [
    "Year",
    "Duration",
    "Votes"
]

categorical_features = [
    "Genre",
    "Director",
    "Actor 1",
    "Actor 2",
    "Actor 3"
]

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=2
        ))
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    max_depth=20
)

In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [ ]:
print("Training model...")

pipeline.fit(X_train, y_train)

print("Model training completed!")

In [ ]:
y_pred = pipeline.predict(X_test)

print("Predictions generated successfully!")

In [ ]:
prediction_df = pd.DataFrame({
    "Actual Rating": y_test.values,
    "Predicted Rating": y_pred
})

prediction_df.head(20)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)

print("Mean Absolute Error:", mae)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Root Mean Squared Error:", rmse)

In [ ]:
r2 = r2_score(y_test, y_pred)

print("R² Score:", r2)

In [ ]:
print("====================================")
print("MODEL PERFORMANCE")
print("====================================")

print(f"Mean Absolute Error : {mae:.4f}")
print(f"Root Mean Squared Error : {rmse:.4f}")
print(f"R² Score : {r2:.4f}")

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_pred, alpha=0.4)

plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.title("Actual vs Predicted IMDb Ratings")

plt.show()

In [ ]:
errors = y_test - y_pred

plt.figure(figsize=(10, 5))

sns.histplot(errors, bins=30, kde=True)

plt.title("Prediction Error Distribution")
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")

plt.show()

In [ ]:
new_movie = pd.DataFrame({
    "Year": [2022],
    "Duration": [120],
    "Votes": [5000],
    "Genre": ["Drama"],
    "Director": ["Example Director"],
    "Actor 1": ["Example Actor 1"],
    "Actor 2": ["Example Actor 2"],
    "Actor 3": ["Example Actor 3"]
})

In [ ]:
predicted_rating = pipeline.predict(new_movie)

print(
    "Predicted IMDb Rating:",
    round(predicted_rating[0], 2)
)

In [ ]:
import joblib

joblib.dump(
    pipeline,
    "movie_rating_prediction_model.pkl"
)

print("Model saved successfully!")

In [ ]:
from google.colab import files

files.download("movie_rating_prediction_model.pkl")

In [ ]:
model_df.to_csv(
    "cleaned_IMDb_movies.csv",
    index=False
)

print("Cleaned dataset saved!")

In [ ]:
files.download("cleaned_IMDb_movies.csv")